# Schema-Constrained Extraction

*Level 9 — Knowledge-Augmented Generation (KAG)*

## Objective

Every prior graph-rag in this repo (Levels 3, 5, 6) lets the LLM invent whatever entity and
relationship types it likes while reading a document. KAG's own departure -- the thing this
level is actually about -- is refusing that freedom: extraction is constrained to a fixed,
closed schema (`Condition`, `Intervention`, `Study`, `Outcome`, `Population` as the only entity
types; five fixed relation types, each locked to one `(subject_type, object_type)` pair), and
anything the model proposes outside that schema is **rejected**, not coerced.

This notebook runs that extraction against real PubMed abstracts (`qiaojin/PubMedQA`,
`pqa_labeled` config) and looks directly at what gets kept and what gets thrown away.

In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))

from kag_common.dataset import prepare
from kag_common.llm import OllamaLLM
from schema.constrained_extraction import extract_from_document
from schema.domain_schema import SchemaValidator, ENTITY_TYPES, RELATION_SCHEMA

print("Entity types:", sorted(ENTITY_TYPES))
print("Relation types:", RELATION_SCHEMA)

Entity types: ['Condition', 'Intervention', 'Outcome', 'Population', 'Study']
Relation types: {'STUDIES': ('Study', 'Condition'), 'USES_INTERVENTION': ('Study', 'Intervention'), 'HAS_POPULATION': ('Study', 'Population'), 'REPORTS_OUTCOME': ('Study', 'Outcome'), 'INTERVENTION_AFFECTS_OUTCOME': ('Intervention', 'Outcome')}


In [2]:
data = prepare(n_documents=6, seed=7)
print(f"Loaded {len(data.corpus)} real PubMed abstracts.")
for doc_id, q in data.questions.items():
    print(f"  {doc_id} | gold={q['answer']:>5} | {q['question'][:80]}")

Loaded 6 real PubMed abstracts.
  24340838 | gold=  yes | Do ventricular arrhythmias in athletes subside over time?
  9722752 | gold=  yes | Does bone anchor fixation improve the outcome of percutaneous bladder neck suspe
  16809243 | gold=   no | Is fetal gender associated with emergency department visits for asthma during pr
  22668712 | gold=   no | Internal derangement of the temporomandibular joint: is there still a place for 
  22617083 | gold=  yes | Does age moderate the effect of personality disorder on coping style in psychiat
  22564465 | gold=  yes | Mammographic screening in Sami speaking municipalities and a control group. Are 


## Extracting from one real abstract

Picking the first document in the sample and looking at its real text before extracting
anything from it.

In [3]:
doc_id = next(iter(data.corpus))
text = data.corpus[doc_id]
print(f"Document {doc_id} ({len(text)} chars):\n")
print(text)

Document 24340838 (1371 chars):

BACKGROUND: Sudden death in athletes can occur during sport activities and is presumably related to ventricular arrhythmias. OBJECTIVES: To investigate the long-term follow-up ofathletes with ventricular arrhythmias during an exercise test. METHODS: From a database of 56,462 athletes we identified 192 athletes (35 years old who had ventricular arrhythmias during an exercise test. Ninety athletes had>or =3 ventricular premature beats (VPB) (group A) and 102 athletes had ventricular couplets or non-sustained ventricular tachycardia during an exercise test (group B). A control group of 92 athletesfrom without ventricular arrhythmias was randomly seleclted from the database (group C). Of the 192 athletes 39 returnied for a repeat exercise test after a mean follow-up period of 70 +/- 25 months and they constitute the study population. RESULTS: Twelve athletes from group A, 21 fromgroup B and 6 from group C returned for a repeat exercise test. The athletes re

In [4]:
llm = OllamaLLM()
validator = SchemaValidator()

entities, relations = extract_from_document(doc_id, text, llm, validator)

print(f"Extracted {len(entities)} entities, {len(relations)} relations from this one document.\n")
for e in entities:
    print(f"  Entity: {e.name!r:35} type={e.type:12} attributes={e.attributes}")
print()
for r in relations:
    print(f"  Relation: {r.subject!r} --{r.relation}--> {r.object!r}")

Extracted 5 entities, 3 relations from this one document.

  Entity: 'Study-24340838'                    type=Study        attributes={'size': 39}
  Entity: 'Condition-Ventricular Arrhythmias' type=Condition    attributes={}
  Entity: 'Group A'                           type=Population   attributes={'size': 90}
  Entity: 'Group B'                           type=Population   attributes={'size': 102}
  Entity: 'Group C'                           type=Population   attributes={'size': 92}

  Relation: 'Study-24340838' --HAS_POPULATION--> 'Group A'
  Relation: 'Study-24340838' --HAS_POPULATION--> 'Group B'
  Relation: 'Study-24340838' --HAS_POPULATION--> 'Group C'


## What gets rejected, and why

The schema is a *closed* vocabulary -- deliberately smaller than what a real abstract usually
mentions (drug dosages, specific statistical tests, journal metadata, ...). Running extraction
over the whole small sample and looking at the validator's own accounting shows the real,
measured cost of that constraint, not an assumed one.

In [5]:
validator_full = SchemaValidator()
all_entities = {}
all_relations = []

for doc_id, text in data.corpus.items():
    entities, relations = extract_from_document(doc_id, text, llm, validator_full)
    all_entities[doc_id] = entities
    all_relations.extend(relations)

print("Validator summary across the sample:")
for k, v in validator_full.summary().items():
    print(f"  {k}: {v}")

Validator summary across the sample:
  accepted_entities: 13
  rejected_entities: 0
  entity_rejection_rate: 0.0
  accepted_relations: 9
  rejected_relations: 5
  relation_rejection_rate: 0.35714285714285715


In [6]:
print(f"Rejection log ({len(validator_full.rejection_log)} entries):")
for line in validator_full.rejection_log[:20]:
    print(" ", line)

Rejection log (8 entries):
  relation rejected: Condition -STUDIES-> Study (expected ('Study', 'Condition'))
  relation rejected: Intervention-Exercise Test -USES_INTERVENTION-> Study-24340838 (unknown entity)
  relation rejected: Condition-Ventricular Arrhythmias -INTERVENTION_AFFECTS_OUTCOME-> Outcome-Lower Peak Heart Rate (unknown entity)
  relation rejected: Intervention-Exercise Test -REPORTS_OUTCOME-> Outcome-Lower Peak Heart Rate (unknown entity)
  doc 16809243: unparseable extraction response
  doc 22668712: unparseable extraction response
  relation rejected: Population-1 -REPORTS_OUTCOME-> Outcome-1 (unknown entity)
  doc 22564465: unparseable extraction response


## Observed result

*(filled in after running this notebook against the real, running Ollama instance -- see the
counts printed above for the actual numbers on this sample.)*